In [ ]:
# 168 Usando RouterChains
from langchain_openai import ChatOpenAI
from langchain.prompts import ChatPromptTemplate, PromptTemplate
from langchain.chains.llm import LLMChain

from langchain.chains.router import MultiPromptChain
from langchain.chains.router.llm_router import LLMRouterChain, RouterOutputParser
from langchain.chains.router.multi_prompt_prompt import MULTI_PROMPT_ROUTER_TEMPLATE

In [ ]:
quimica_template = ChatPromptTemplate.from_template("""Você é um químico muito experiente.
Você é excelente em responder perguntas sobre química de forma clara e objetiva.
Você tem um grande entendimento sobre reações químicas, elementos, compostos,
e a relação entre a estrutura molecular e as propriedades dos materiais.
Quando você não sabe a resposta para uma pergunta, você admite que não sabe.

Aqui está uma pergunta: {input}""")

geografia_template = ChatPromptTemplate.from_template("""Você é um geógrafo muito bem informado.
Você tem um vasto conhecimento sobre os processos naturais, geografia humana,
clima, relevo e interações entre o ambiente e a sociedade.
Você é habilidoso em explicar como fatores físicos e humanos afetam
o mundo ao nosso redor.

Aqui está uma pergunta: {input}""")

biologia_template = ChatPromptTemplate.from_template("""Você é um biólogo muito capacitado.
Você tem um grande conhecimento sobre os seres vivos, suas estruturas, funções
e a interação entre diferentes organismos e seus ambientes.
Você é excelente em explicar conceitos de biologia de maneira clara,
tanto para iniciantes quanto para estudantes avançados.

Aqui está uma pergunta: {input}""")

In [ ]:
prompt_infos = [
    {
        "name": "Química",
        "description": "Ideal para responder pergunta de química",
        "prompt_template": quimica_template
    },
    {
        "name": "Geografia",
        "description": "Ideal para responder pergunta de geografia",
        "prompt_template": geografia_template
    },
    {
        "name": "Biologia",
        "description": "Ideal para responder pergunta de biologia",
        "prompt_template": biologia_template
    },
]

In [ ]:
chat = ChatOpenAI(model="gpt-3.5-turbo-0125")

chains_destino = {}
for info in prompt_infos:
    chain = LLMChain(llm=chat, prompt=info["prompt_template"], verbose=True)
    chains_destino[info["name"]] = chain

chains_destino

## Resultado
{
    'Química': LLMChain(verbose=True, prompt=ChatPromptTemplate(input_variables=['input'], messages=[HumanMessagePromptTemplate(...)])),
    'Geografia': LLMChain(verbose=True, prompt=ChatPromptTemplate(input_variables=['input'], messages=[HumanMessagePromptTemplate(...)])),
    'Biologia': LLMChain(verbose=True, prompt=ChatPromptTemplate(input_variables=['input'], messages=[HumanMessagePromptTemplate(...)]))
}

In [ ]:
## para todos os nomes e descrições que estao nos prompts infos, exiba conforme abaixo.
destinos = [f'{p["name"]}: {p["description"]}' for p in prompt_infos]
destinos_str = "\n".join(destinos)
print(destinos_str)

Resultado
Química: Ideal para responder pergunta de química
Geografia: Ideal para responder pergunta de geografia
Biologia: Ideal para responder pergunta de biologia

In [ ]:
## MULTI_PROMPT_ROUTER_TEMPLATE é um template padrão do LangChain usado 
# para criar um prompt que orienta o modelo a escolher entre múltiplos destinos 
# (neste caso, Química, Geografia, Biologia).
# format(destinations=destinos_str) insere a lista de destinos 
# (que você gerou anteriormente com destinos_str) dentro do template.
# O resultado (router_template) é um prompt completo que será usado pelo 
# roteador LLM para decidir qual cadeia (LLMChain) deve responder à pergunta do usuário.
router_template = MULTI_PROMPT_ROUTER_TEMPLATE.format(
    destinations=destinos_str)
print(router_template)


In [ ]:
###Esse código monta um sistema inteligente que:
# Recebe uma pergunta.
# Usa o roteador para decidir qual especialista (cadeia) deve responder.
# Se não encontrar correspondência, usa a cadeia padrão.
# Essa é a principal aplicabilidade do chain.
# Encaminhar para a chain especifica de acordo com o tipo de prompt.

router_template = PromptTemplate(
    template=router_template,
    input_variables=["input"],
    output_parser=RouterOutputParser()
)

router_chain = LLMRouterChain.from_llm(chat, router_template, verbose=True)

default_prompt = ChatPromptTemplate.from_template("{input}")
default_chain = LLMChain(llm=chat, prompt=default_prompt, verbose=True)
chain = MultiPromptChain(
    router_chain=router_chain,
    destination_chains=chains_destino,
    default_chain=default_chain,
    verbose=True
)

In [ ]:
chain.invoke("input": "O que é o El Nino?")

In [ ]:
chain.invoke("input": "Para que serve os cromossomos?")